#  NFL Big Data Bowl 2025 Analysis

This notebook analyzes NFL game data, focusing on games 1 to 8 of the  San Francisco 49ers (SF). It includes data loading, preprocessing, feature engineering, and a predictive model with analyses for run v pass outcomes. Additional analyses are then performed by removing features the defense wouldn't know prior to the snap.

In [ ]:
# NFL Big Data Bowl 2025 - Run vs Pass Prediction (Enhanced)
from scipy.spatial import cKDTree
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import xgboost as xgb



# Load necessary datasets
project_dir = '/kaggle/input/nfl-big-data-bowl-2025'
games = pd.read_csv(f'{project_dir}/games.csv')
plays = pd.read_csv(f'{project_dir}/plays.csv')
players = pd.read_csv(f'{project_dir}/players.csv')
player_play = pd.read_csv(f'{project_dir}/player_play.csv')
tracking_week_1 = pd.read_csv(f'{project_dir}/tracking_week_1.csv')
tracking_week_2 = pd.read_csv(f'{project_dir}/tracking_week_2.csv')

# Convert 'gameDate' to datetime format
games['gameDate'] = pd.to_datetime(games['gameDate'])

# Filter for Week 1 game between San Francisco 49ers and Chicago Bears
sf_chi_games = games[
    (
        ((games['homeTeamAbbr'] == 'SF') & (games['visitorTeamAbbr'] == 'CHI')) |
        ((games['homeTeamAbbr'] == 'CHI') & (games['visitorTeamAbbr'] == 'SF'))
    ) &
    (games['week'] == 1)
]

# Filter for Week 2 game between San Francisco 49ers and Seattle Seahawks on 09/18/2022
sf_sea_games = games[
    (
        ((games['homeTeamAbbr'] == 'SF') & (games['visitorTeamAbbr'] == 'SEA')) |
        ((games['homeTeamAbbr'] == 'SEA') & (games['visitorTeamAbbr'] == 'SF'))
    ) &
    (games['week'] == 2) &
    (games['gameDate'] == '2022-09-18')
]

# Filter for Week 3 game between San Francisco 49ers and Denver Broncos on 09/25/2022
sf_den_games = games[
    (
        ((games['homeTeamAbbr'] == 'SF') & (games['visitorTeamAbbr'] == 'DEN')) |
        ((games['homeTeamAbbr'] == 'DEN') & (games['visitorTeamAbbr'] == 'SF'))
    ) &
    (games['week'] == 3) &
    (games['gameDate'] == '2022-09-25')
]

# Filter for Week 4 game between San Francisco 49ers and Los Angeles Rams on 10/03/2022
sf_lar_games_w4 = games[
    (
        ((games['homeTeamAbbr'] == 'SF') & (games['visitorTeamAbbr'] == 'LAR')) |
        ((games['homeTeamAbbr'] == 'LAR') & (games['visitorTeamAbbr'] == 'SF'))
    ) &
    (games['week'] == 4) &
    (games['gameDate'] == '2022-10-03')
]

# Filter for Week 5 game between San Francisco 49ers and Carolina Panthers on 10/09/2022
sf_car_games = games[
    (
        ((games['homeTeamAbbr'] == 'SF') & (games['visitorTeamAbbr'] == 'CAR')) |
        ((games['homeTeamAbbr'] == 'CAR') & (games['visitorTeamAbbr'] == 'SF'))
    ) &
    (games['week'] == 5) &
    (games['gameDate'] == '2022-10-09')
]

# Filter for Week 6 game between San Francisco 49ers and Atlanta Falcons on 10/16/2022
sf_atl_games = games[
    (
        ((games['homeTeamAbbr'] == 'SF') & (games['visitorTeamAbbr'] == 'ATL')) |
        ((games['homeTeamAbbr'] == 'ATL') & (games['visitorTeamAbbr'] == 'SF'))
    ) &
    (games['week'] == 6) &
    (games['gameDate'] == '2022-10-16')
]

# Filter for Week 7 game between San Francisco 49ers and Kansas City Chiefs on 10/23/2022
sf_kc_games = games[
    (
        ((games['homeTeamAbbr'] == 'SF') & (games['visitorTeamAbbr'] == 'KC')) |
        ((games['homeTeamAbbr'] == 'KC') & (games['visitorTeamAbbr'] == 'SF'))
    ) &
    (games['week'] == 7) &
    (games['gameDate'] == '2022-10-23')
]

# Filter for Week 8 game between San Francisco 49ers and Los Angeles Rams on 10/30/2022
sf_lar_games_w8 = games[
    (
        ((games['homeTeamAbbr'] == 'SF') & (games['visitorTeamAbbr'] == 'LAR')) |
        ((games['homeTeamAbbr'] == 'LAR') & (games['visitorTeamAbbr'] == 'SF'))
    ) &
    (games['week'] == 8) &
    (games['gameDate'] == '2022-10-30')
]

# Combine gameIds
selected_games = pd.concat([
    sf_chi_games, sf_sea_games, sf_den_games, sf_lar_games_w4,
    sf_car_games, sf_atl_games, sf_kc_games, sf_lar_games_w8
])
selected_game_ids = selected_games['gameId'].tolist()

# Filter plays and player_play data for selected games
plays = plays[plays['gameId'].isin(selected_game_ids)]
player_play = player_play[player_play['gameId'].isin(selected_game_ids)]

# Load and filter tracking data for weeks 1 through 8
tracking_data_list = []
for week in range(1, 9):
    tracking_week = pd.read_csv(f'{project_dir}/tracking_week_{week}.csv')
    tracking_week_filtered = tracking_week[tracking_week['gameId'].isin(selected_game_ids)]
    tracking_data_list.append(tracking_week_filtered)

# Combine tracking data and filter for pre-snap
tracking_data = pd.concat(tracking_data_list, ignore_index=True)
tracking_data = tracking_data[tracking_data['frameType'] == 'BEFORE_SNAP']



# Merge games and plays data
plays = plays.merge(
    games[['gameId', 'homeTeamAbbr', 'visitorTeamAbbr']],
    on='gameId', how='left'
)

# Create target variable 'playType' (run or pass)
plays['playType'] = plays['isDropback'].map({True: 'pass', False: 'run'})
plays = plays[plays['playType'].isin(['run', 'pass'])]

# Feature Engineering

# 1. Game situation features
plays['score_difference'] = plays['preSnapHomeScore'] - plays['preSnapVisitorScore']

def parse_game_clock(clock_str):
    minutes, seconds = map(int, clock_str.split(':'))
    return minutes * 60 + seconds

plays['seconds_remaining'] = plays.apply(lambda row: (4 - row['quarter']) * 15 * 60 + parse_game_clock(row['gameClock']), axis=1)

# 2. Player formation features
formation_dummies = pd.get_dummies(plays['offenseFormation'], prefix='formation')
plays = pd.concat([plays, formation_dummies], axis=1)

# 3. Player motion features
motion_features = player_play.groupby(['gameId', 'playId']).agg({
    'inMotionAtBallSnap': 'sum',
    'shiftSinceLineset': 'sum',
    'motionSinceLineset': 'sum'
}).reset_index()

plays = plays.merge(motion_features, on=['gameId', 'playId'], how='left')

# 4. Defensive coverage features
coverage_dummies = pd.get_dummies(plays['pff_passCoverage'], prefix='coverage')
plays = pd.concat([plays, coverage_dummies], axis=1)

# 5. Pre-snap player positions
def get_presnap_positions(group):
    presnap = group[group['frameType'] == 'BEFORE_SNAP']
    if presnap.empty:
        return pd.Series({'presnap_offense_x_mean': np.nan, 'presnap_offense_y_mean': np.nan,
                          'presnap_defense_x_mean': np.nan, 'presnap_defense_y_mean': np.nan})
    
    # Get the first gameId and playId from the group
    game_id = group['gameId'].iloc[0]
    play_id = group['playId'].iloc[0]
    
    # Use .loc to select the possession team
    possession_team = plays.loc[(plays['gameId'] == game_id) & (plays['playId'] == play_id), 'possessionTeam'].iloc[0]
    
    presnap['teamType'] = np.where(presnap['club'] == possession_team, 'offense', 'defense')
    
    offense = presnap[presnap['teamType'] == 'offense']
    defense = presnap[presnap['teamType'] == 'defense']
    
    return pd.Series({
        'presnap_offense_x_mean': offense['x'].mean(),
        'presnap_offense_y_mean': offense['y'].mean(),
        'presnap_defense_x_mean': defense['x'].mean(),
        'presnap_defense_y_mean': defense['y'].mean()
    })


presnap_positions = tracking_data.groupby(['gameId', 'playId']).apply(get_presnap_positions).reset_index()
plays = plays.merge(presnap_positions, on=['gameId', 'playId'], how='left')

# 6. Merge tracking_data with plays to get homeTeamAbbr and visitorTeamAbbr
tracking_data = tracking_data.merge(
    plays[['gameId', 'playId', 'possessionTeam', 'homeTeamAbbr', 'visitorTeamAbbr']],
    on=['gameId', 'playId'],
    how='left'
)

# Identify team type
def identify_team_type(row):
    if pd.isna(row['nflId']):
        return 'football'
    elif row['club'] == row['possessionTeam']:
        return 'offense'
    else:
        return 'defense'

tracking_data['teamType'] = tracking_data.apply(identify_team_type, axis=1)


# Determine offense or defense
def is_offense(row):
    return 1 if row['teamType'] == 'offense' else 0

tracking_data['offenseDefense'] = tracking_data.apply(is_offense, axis=1)

def nearest_opponent_distance(group):
    offense = group[group['teamType'] == 'offense']
    defense = group[group['teamType'] == 'defense']
    tree = cKDTree(defense[['x', 'y']].values)
    distances, _ = tree.query(offense[['x', 'y']].values, k=1)
    return pd.Series({'nearestDefenderDist': distances.min()})

tracking_data['dir_rad'] = np.deg2rad(tracking_data['dir'])
tracking_data['vx'] = tracking_data['s'] * np.cos(tracking_data['dir_rad'])
tracking_data['vy'] = tracking_data['s'] * np.sin(tracking_data['dir_rad'])
tracking_data['speed'] = tracking_data['s']
tracking_data['acceleration'] = tracking_data['a']

nearest_distances = tracking_data.groupby(['gameId', 'playId', 'frameId']).apply(nearest_opponent_distance)
nearest_distances = nearest_distances.reset_index(level=[0,1,2], drop=True)
tracking_data = tracking_data.join(nearest_distances)

agg_features = tracking_data.groupby(['gameId', 'playId', 'nflId']).agg({
    'speed': ['max', 'mean'],
    'acceleration': ['max', 'mean'],
    'nearestDefenderDist': ['min', 'mean'],
    'x': ['min', 'max'],
    'y': ['min', 'max']
}).reset_index()

agg_features.columns = [
    'gameId', 'playId', 'nflId',
    'speed_max', 'speed_mean',
    'acc_max', 'acc_mean',
    'dist_min', 'dist_mean',
    'x_min', 'x_max',
    'y_min', 'y_max'
]

agg_features['x_range'] = agg_features['x_max'] - agg_features['x_min']
agg_features['y_range'] = agg_features['y_max'] - agg_features['y_min']

agg_features = agg_features.merge(players[['nflId', 'position']], on='nflId', how='left')

plays = plays.merge(agg_features, on=['gameId', 'playId'], how='left')

# Prepare features for modeling
all_columns = plays.columns
base_features = [
    'down', 'yardsToGo', 'absoluteYardlineNumber', 'preSnapHomeTeamWinProbability',
    'playClockAtSnap', 'score_difference', 'seconds_remaining',
    'inMotionAtBallSnap', 'shiftSinceLineset', 'motionSinceLineset',
    'presnap_offense_x_mean', 'presnap_offense_y_mean',
    'presnap_defense_x_mean', 'presnap_defense_y_mean',
    'speed_max', 'speed_mean', 'acc_max', 'acc_mean',
    'dist_min', 'dist_mean', 'x_range', 'y_range'
]

formation_dummies = [col for col in all_columns if col.startswith('formation_')]
coverage_dummies = [col for col in all_columns if col.startswith('coverage_')]
position_dummies = pd.get_dummies(plays['position'], prefix='position')
plays = pd.concat([plays, position_dummies], axis=1)
position_dummies = [col for col in plays.columns if col.startswith('position_')]

all_feature_columns = base_features + formation_dummies + coverage_dummies + position_dummies

# Check which columns actually exist in the DataFrame and have non-null values
existing_columns = plays[all_feature_columns].columns[plays[all_feature_columns].notna().any()].tolist()

print(f"Existing columns: {existing_columns}")

X = plays[existing_columns]
y = plays['playType'].map({'run': 0, 'pass': 1})

# Handle missing values
imputer = SimpleImputer(strategy='mean')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Normalize features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X_imputed.columns)

# Use RFE with XGBoost to find the optimal subset of features
def rfe_feature_selection(X, y, n_features_to_select=1):
    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    rfe_selector = RFE(estimator=model, n_features_to_select=n_features_to_select, step=1)
    rfe_selector = rfe_selector.fit(X, y)
    return rfe_selector

# Perform RFE
max_features = len(X_scaled.columns)
best_score = 0
best_n_features = 0

for n in range(1, max_features + 1):
    rfe_selector = rfe_feature_selection(X_scaled, y, n_features_to_select=n)
    selected_features = X_scaled.columns[rfe_selector.support_].tolist()
    X_subset = X_scaled[selected_features]
    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_subset, y, cv=cv, scoring='roc_auc')
    mean_score = scores.mean()
    if mean_score > best_score:
        best_score = mean_score
        best_n_features = n


print(f"Optimal number of features: {best_n_features}")
print(f"Best ROC AUC score: {best_score:.4f}")

# Select the best features
rfe_selector = rfe_feature_selection(X_scaled, y, best_n_features)
selected_features = X_scaled.columns[rfe_selector.support_].tolist()
X_selected = X_scaled[selected_features]

# Combine X_selected and y into a single DataFrame
combined_df = pd.concat([X_selected, y], axis=1)
combined_df.rename(columns={'playType': 'playType'}, inplace=True)

# Check for duplicate rows
duplicate_rows = combined_df[combined_df.duplicated()]

print(f"Number of duplicate rows: {len(duplicate_rows)}")
print("\nFirst few duplicate rows (if any):")
print(duplicate_rows.head())

# Remove duplicates
combined_df_unique = combined_df.drop_duplicates()

# Separate features and target
X_selected = combined_df_unique.drop(columns=['playType'])
y = combined_df_unique['playType']

print(f"Rows before removing duplicates: {len(combined_df)}")
print(f"Rows after removing duplicates: {len(combined_df_unique)}")

# Optuna hyperparameter optimization
def objective(trial):
    param = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }
    model = xgb.XGBClassifier(**param, use_label_encoder=False, eval_metric='logloss')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_selected, y, cv=cv, scoring='roc_auc')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

best_params = study.best_params
print("Best hyperparameters:", best_params)

# Train final model with best hyperparameters
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)
final_model = xgb.XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss')
final_model.fit(X_train, y_train)

# Predict and evaluate
y_pred = final_model.predict(X_test)
y_pred_proba = final_model.predict_proba(X_test)[:, 1]

print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC AUC:', roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))

# Plot Feature Importances
plt.figure(figsize=(12, 8))
xgb.plot_importance(final_model, max_num_features=20)
plt.title('Feature Importances for Run vs Pass Prediction')
plt.tight_layout()
plt.show()

# Analyze impact of top features on outcomes
# feature_importance = final_model.feature_importances_
# top_features = [selected_features[i] for i in np.argsort(feature_importance)[-5:]]
# for feature in top_features:
#     plt.figure(figsize=(6, 4))
#     sns.boxplot(x=y_test, y=X_test[feature])
#     plt.title(f'Impact of {feature} on Play Type')
#     plt.xlabel('Play Type')
#     plt.ylabel(feature)
#     plt.show()

# Correlation analysis
correlation_matrix = X_selected.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Additional analysis: Learning curve
from sklearn.model_selection import learning_curve

train_sizes, train_scores, test_scores = learning_curve(
    final_model, X_selected, y, cv=5, n_jobs=-1, 
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='roc_auc'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, label='Training score', color='blue', marker='o')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
plt.plot(train_sizes, test_mean, label='Cross-validation score', color='red', marker='o')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.15, color='red')
plt.title('Learning Curve')
plt.xlabel('Training Examples')
plt.ylabel('ROC AUC Score')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# Confusion Matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# ROC Curve
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc_score(y_test, y_pred_proba):.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.show()

# Feature importance bar plot
plt.figure(figsize=(12, 8))
feature_importance = final_model.feature_importances_
sorted_idx = np.argsort(feature_importance)
pos = np.arange(sorted_idx.shape[0]) + .5
plt.barh(pos, feature_importance[sorted_idx], align='center')
plt.yticks(pos, np.array(selected_features)[sorted_idx])
plt.xlabel('Feature Importance')
plt.title('Feature Importance (MDI)')
plt.tight_layout()
plt.show()

print("Analysis complete.")

print("Additional analyses based on refined feature set:")

# Get the top features from RFE results that defense would know prior to snap
top_features = [
    feature for feature in selected_features 
    if feature not in ['preSnapHomeTeamWinProbability', 'speed_mean']
][:14]

print(f"Selected top features (excluding specified features):\n{top_features}")

# Create a new dataset with only the selected features
X_refined = X_selected[top_features]

# Train a new model with the refined feature set
X_train_refined, X_test_refined, y_train, y_test = train_test_split(X_refined, y, test_size=0.2, random_state=42)
refined_model = xgb.XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss')
refined_model.fit(X_train_refined, y_train)

# Predict and evaluate
y_pred_refined = refined_model.predict(X_test_refined)
y_pred_proba_refined = refined_model.predict_proba(X_test_refined)[:, 1]

print('Refined Model Accuracy:', accuracy_score(y_test, y_pred_refined))
print('Refined Model ROC AUC:', roc_auc_score(y_test, y_pred_proba_refined))
print('Refined Model Classification Report:')
print(classification_report(y_test, y_pred_refined))